# 05 Grad-CAM Baseline

Grad-CAM baseline for qualitative comparison against SODT explanations.

This notebook generates Grad-CAM heatmaps from the Faster R-CNN backbone's
ResNet-50 `layer4` (c5 stage) and visualises them alongside each detection.
The resulting saliency maps should highlight defect-relevant image regions,
providing a standard XAI reference point to validate the fidelity of the
SODT's intrinsic symbolic explanations.

**Reference:** Selvaraju et al., *Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization*, ICCV 2017.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image

from notebooks.util import resolve_root
from neuro.config import NeuroConfig, NeuroTrainConfig
from neuro.inference import load_checkpoint_model
from neuro.preprocess_dataset import test_preprocess
from neuro.prepare_dataset import PCBDataset
from gradcam.gradcam import GradCAM
from gradcam.visualize import gradcam_for_detections, plot_gradcam_comparison
from util.config import load_yaml
from util.device import select_device

PROJECT_ROOT = resolve_root()

## 1. Load Model & Config

In [ ]:
train_config = load_yaml(Path("neuro_train.yaml"), NeuroTrainConfig)
model_config = load_yaml(Path("neuro.yaml"), NeuroConfig)

device = select_device(train_config["device"])
checkpoint_path = PROJECT_ROOT / "checkpoints" / "neuro" / "BESTEST.pt"

model, checkpoint = load_checkpoint_model(
    checkpoint_path,
    model_config_path=Path("neuro.yaml"),
    train_config_path=Path("neuro_train.yaml"),
    device=str(device),
)

class_names = tuple(train_config["dataset"]["class_names"])
preprocess = test_preprocess()

print(f"Model loaded on {device}")
print(f"Classes: {class_names}")

## 2. Initialise Grad-CAM

In [ ]:
gradcam = GradCAM(model, device=str(device))
print("Grad-CAM initialised — hooked into ResNet-50 layer4 (c5)")

## 3. Pick Random Test Samples

In [ ]:
import random

test_dataset = PCBDataset(train_config, split_file="test.txt")
test_dataset.add_preprocess(preprocess)

N = 4  # number of test images to visualise
random.seed(42)
sample_indices = random.sample(range(len(test_dataset)), k=min(N, len(test_dataset)))

print(f"Selected {len(sample_indices)} test samples: {sample_indices}")

## 4. Run Inference + Grad-CAM

In [ ]:
SCORE_THRESHOLD = 0.3

for idx in sample_indices:
    image_tensor, target = test_dataset[idx]

    # Run standard inference to get detection boxes.
    with torch.inference_mode():
        predictions = model([image_tensor.to(device)])
        prediction = {k: v.detach().cpu() for k, v in predictions[0].items()}

    # Generate Grad-CAM heatmaps.
    results = gradcam_for_detections(
        gradcam, image_tensor, prediction,
        score_threshold=SCORE_THRESHOLD,
        output_size=(7, 7),
    )

    fig = plot_gradcam_comparison(
        image_tensor, results, class_names,
    )
    fig.suptitle(f"Test sample #{idx}", fontsize=14, fontweight="bold", y=1.01)
    plt.show()

print("Done.")

## 5. Cleanup

In [ ]:
gradcam.release()
print("Grad-CAM hooks released.")